# 03 — SQAPT and CT-PIMC: The Best Quantum Analog Methods

**What you'll learn:**
- What parallel tempering is and why it helps
- How the (β, Γ) ladder works in SQAPT
- How to design a ladder with `auto_ladder_sqa_tuned`
- How to monitor replica swap acceptance
- What CT-PIMC is and when to prefer it
- Head-to-head comparison: SA vs SQA vs SQAPT vs CT-PIMC

In [ ]:
import os, sys
_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
_PY   = os.path.join(_ROOT, 'python')
if _PY not in sys.path:
    sys.path.insert(0, _PY)

import numpy as np
import matplotlib.pyplot as plt
from qanneal import (
    DenseIsing,
    solve, auto_schedule_sa_tuned, auto_schedule_sqa_tuned,
    auto_ladder_sqa_tuned,
    SQAParallelTemperingAnnealer, CTPIMCAnnealer,
)

---

## 1 — Parallel Tempering: The Key Idea

**Problem with standard SQA**: one annealing trajectory — once stuck in a local minimum,
the schedule doesn't restart.

**Parallel tempering (PT) / replica exchange** runs **many copies** (replicas) of the system
simultaneously at *different* (β, Γ) points:

```
Replica 0: (β=0.1, Γ=5.0)  ← hot, high tunneling — explores freely
Replica 1: (β=0.5, Γ=2.0)
Replica 2: (β=1.5, Γ=0.8)
Replica 3: (β=4.0, Γ=0.2)
...                         ← cold, low tunneling — refines good solutions
Replica 7: (β=8.0, Γ=0.01)
```

Periodically, **adjacent replicas propose to swap** their spin configurations.
A good solution discovered at high (β, Γ) can "flow" to colder replicas for refinement.
A stuck replica at low temperature can "escape" by receiving a hot configuration.

**Swap acceptance**: proportional to the energy difference between configurations at the two temperatures.
Acceptance rate of 20–50% is ideal — too high means the ladder is too dense, too low means it's too sparse.

---

## 2 — Test Problem: Random Spin Glass

Random ±J couplings (spin glass) have many metastable states — ideal for testing PT.

In [ ]:
def make_spin_glass(n, density=0.4, seed=0):
    """Random ±1 couplings on a random graph."""
    rng = np.random.default_rng(seed)
    h = rng.uniform(-0.5, 0.5, n)
    J = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            if rng.random() < density:
                v = float(rng.choice([-1.0, 1.0]))
                J[i, j] = J[j, i] = v
    return DenseIsing(h, J)

n = 20
ising = make_spin_glass(n, density=0.45, seed=99)

# Brute-force ground state (feasible for n=20)
best_E = float('inf')
for mask in range(1 << n):
    spins = [1 if (mask >> i) & 1 else -1 for i in range(n)]
    E = ising.energy(spins)
    if E < best_E:
        best_E = E

print(f'Spin glass n={n}: ground state energy = {best_E:.4f}')

---

## 3 — Building a (β, Γ) Ladder

`auto_ladder_sqa_tuned` builds a physically sensible ladder:
- β ladder: geometric from β_min (high T, high tunneling) to β_max (low T, frozen)
- Γ ladder: **inverse** geometric — high Γ paired with low β, low Γ paired with high β

In [ ]:
ladder_8  = auto_ladder_sqa_tuned(ising, replicas=8,  mode='balanced')
ladder_16 = auto_ladder_sqa_tuned(ising, replicas=16, mode='balanced')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ladder, label, ms in [(ladder_8, '8 replicas', 8), (ladder_16, '16 replicas', 5)]:
    axes[0].plot(ladder.betas, ladder.gammas, 'o-', ms=ms, label=label)
axes[0].set_xlabel('β (inverse temperature)')
axes[0].set_ylabel('Γ (transverse field)')
axes[0].set_title('(β, Γ) ladder — each dot is a replica')
axes[0].legend()

# Show how the ladder covers the (β, Γ) phase space
rungs = list(range(8))
width = 0.35
axes[1].bar([r - width/2 for r in rungs], ladder_8.betas,  width, label='β', color='#4c8bf5')
ax_r = axes[1].twinx()
ax_r.bar([r + width/2 for r in rungs], ladder_8.gammas, width, label='Γ', color='#c84b31', alpha=0.7)
axes[1].set_xlabel('Replica index')
axes[1].set_ylabel('β', color='#4c8bf5')
ax_r.set_ylabel('Γ', color='#c84b31')
axes[1].set_title('Ladder values per replica (8-rung ladder)')
axes[1].set_xticks(rungs)

fig.tight_layout()
plt.show()

---

## 4 — SQAPT: Low-Level Usage with Swap Acceptance Monitoring

In [ ]:
from qanneal import SQAParallelTemperingAnnealer

ladder = auto_ladder_sqa_tuned(ising, replicas=8, mode='balanced')

annealer = SQAParallelTemperingAnnealer(
    ising,
    ladder.betas,    # β ladder
    ladder.gammas,   # Γ ladder
    trotter_slices=24,
)
annealer.set_seed(42)

result = annealer.run(
    sweeps_per_step=50,       # local SQA sweeps per epoch
    worldline_sweeps=5,
    steps=80,                 # number of swap epochs
    swap_interval=1,          # attempt swaps every epoch
    cluster_sweeps=1,
)

print(f'SQAPT best energy: {result.best_energy:.4f}  (ground state: {best_E:.4f})')
print(f'Gap to ground state: {result.best_energy - best_E:.4f}')
print(f'Final replica energies: {[f"{e:.3f}" for e in result.final_energies]}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Energy trace
ax = axes[0]
ax.plot(result.average_energy_trace, color='#4c8bf5', lw=1.5)
ax.axhline(best_E, color='green', linestyle='--', lw=1.5, label=f'Ground state ({best_E:.3f})')
ax.axhline(result.best_energy, color='#c84b31', linestyle=':', lw=1.5, label=f'SQAPT best ({result.best_energy:.3f})')
ax.set_xlabel('PT epoch')
ax.set_ylabel('Average energy')
ax.set_title('SQAPT energy convergence')
ax.legend()

# Swap acceptance
ax2 = axes[1]
ax2.plot(result.swap_acceptance_trace, color='#c84b31', lw=1.5)
ax2.axhline(0.5, color='green', linestyle='--', lw=1, label='Ideal upper bound (0.5)')
ax2.axhline(0.2, color='orange', linestyle='--', lw=1, label='Ideal lower bound (0.2)')
ax2.set_xlabel('PT epoch')
ax2.set_ylabel('Swap acceptance fraction')
ax2.set_title('Replica swap acceptance rate')
ax2.set_ylim(0, 1)
ax2.legend()

mean_swap = np.mean(result.swap_acceptance_trace)
print(f'Mean swap acceptance: {mean_swap:.3f}  (target: 0.20–0.50)')
if mean_swap < 0.1:
    print('  → Too low: make ladder denser (more replicas, smaller gaps)')
elif mean_swap > 0.6:
    print('  → Too high: ladder might be too dense; fewer replicas could work')
else:
    print('  → Good acceptance rate!')

fig.tight_layout()
plt.show()

---

## 5 — CT-PIMC: Continuous-Time Path Integral

### Physics
SQA uses a **discrete** Trotter decomposition (M slices). CT-PIMC uses **continuous** imaginary time:
- No Trotter discretisation error.
- Worldlines are continuous curves; kinks (spin flips) can occur at any time.
- Uses Swendsen–Wang cluster updates over worldline segments.

### When to prefer CT-PIMC
- When you want a closer analog to quantum sampling (density-matrix level)
- For problems where Trotter error is visible (large β × Γ product)
- The `gamma_end_scale=0.04` setting in `auto_schedule_sqa_tuned` ensures Γ → 0 for clean classical projection

In [ ]:
# CT-PIMC via the high-level solve() interface
ctpimc_result = solve(
    ising,
    method='ctpimc',
    reads=30,
    sweeps_per_beta=80,
    seed=42,
    progress=False,
)
print(f'CT-PIMC best energy: {ctpimc_result.best_energy:.4f}  (ground state: {best_E:.4f})')

In [ ]:
# CT-PIMC via low-level API for schedule control
from qanneal import CTPIMCAnnealer, SQASchedule

# Note: gamma must end very close to 0 for clean classical projection
schedule_ct = SQASchedule.from_vectors(
    betas  = np.linspace(0.1, 6.0, 60).tolist(),
    gammas = np.geomspace(5.0, 0.001, 60).tolist(),  # very low gamma_end!
)

ct_annealer = CTPIMCAnnealer(ising, schedule_ct, qubits_per_update=1, qubits_per_chain=1)
ct_annealer.set_seed(0)
ct_res = ct_annealer.run(sweeps_per_beta=60, reads=4)

print(f'CT-PIMC (low-level) best energy: {ct_res.best_energy:.4f}')
print(f'Energy trace length: {len(ct_res.energy_trace)} steps')

---

## 6 — Full Comparison: SA vs SQA vs SQAPT vs CT-PIMC

In [ ]:
READS = 50
SEED  = 7

methods = [
    ('sa',     'SA',      dict(sweeps_per_beta=60)),
    ('sqa',    'SQA',     dict(sweeps_per_beta=60, worldline_sweeps=5, trotter_slices=24)),
    ('sqapt',  'SQAPT',   dict(sweeps_per_beta=50, worldline_sweeps=5, trotter_slices=24,
                                replicas=8, pt_steps=70)),
    ('ctpimc', 'CT-PIMC', dict(sweeps_per_beta=80)),
]

all_results = {}
for method, label, kwargs in methods:
    r = solve(
        ising,
        method=method,
        reads=READS,
        seed=SEED,
        progress=False,
        **kwargs,
    )
    all_results[label] = r
    hit = np.mean([abs(e - best_E) < 0.5 for e in r.energies])
    gap = r.best_energy - best_E
    print(f'{label:10s}: best={r.best_energy:.4f}  gap={gap:.4f}  hit={hit:.2f}  mean={np.mean(r.energies):.3f}')

print(f'\nGround state: {best_E:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

labels = list(all_results.keys())
colors = ['#4c8bf5', '#f5a623', '#c84b31', '#2ca02c']

# 1. Best energy
ax = axes[0]
best_Es = [all_results[l].best_energy for l in labels]
bars = ax.bar(labels, best_Es, color=colors)
ax.axhline(best_E, color='green', linestyle='--', lw=2, label=f'Ground state ({best_E:.3f})')
ax.set_ylabel('Best energy found')
ax.set_title('Best energy (lower = better)')
ax.legend()

# 2. Hit rate
ax2 = axes[1]
hit_rates = [np.mean([abs(e - best_E) < 0.5 for e in all_results[l].energies]) for l in labels]
ax2.bar(labels, hit_rates, color=colors)
ax2.set_ylabel(f'Hit rate (gap < 0.5)')
ax2.set_title(f'Ground-state hit rate ({READS} reads)')
ax2.set_ylim(0, 1.05)

# 3. Energy distribution
ax3 = axes[2]
all_E_min = min(e for l in labels for e in all_results[l].energies)
all_E_max = max(e for l in labels for e in all_results[l].energies)
bins = np.linspace(all_E_min - 0.5, all_E_max + 0.5, 25)
for label, color in zip(labels, colors):
    ax3.hist(all_results[label].energies, bins=bins, alpha=0.6, label=label, color=color)
ax3.axvline(best_E, color='green', linestyle='--', lw=2, label='Ground state')
ax3.set_xlabel('Energy')
ax3.set_ylabel('Count')
ax3.set_title('Energy distribution')
ax3.legend(fontsize=8)

fig.suptitle(f'Spin glass n={n}: method comparison ({READS} reads each)', fontsize=12)
fig.tight_layout()
plt.show()

---

## 7 — When to Use Each Method

| Method | Best for | Avoid when |
|--------|---------|----------|
| **SA** | Fast prototyping, simple landscapes, baseline | Rugged multi-modal energy landscapes |
| **SQA** | Moderate difficulty, barriers that SA misses | Very large n (memory: n × slices) |
| **SQAPT** | Rugged/multi-modal problems, production runs | Very tight time budgets (more compute than SQA) |
| **CT-PIMC** | Research, D-Wave-like statistics, no Trotter error | Very small Γ (kink density → 0; CT update may stall) |

**Default recommendation**: start with `sqapt`, `replicas=8`, `mode="balanced"`.

**Next**: `04_large_problems.ipynb` — SparseIsing, HPC scaling, and SLURM deployment.